In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import pickle

df = pd.read_csv('data.csv') 
df = df.drop(columns=['id'], errors='ignore')
df['diagnosis'] = df['diagnosis'].map({'M': 1, 'B': 0})

top_features = df.corr()['diagnosis'].sort_values(ascending=False).index[1:6].tolist()
print(f"Your Top 5 Selection Criteria: {top_features}")

X = df[top_features]
y = df['diagnosis']

# ده كله لتدريب الموديل
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression()
model.fit(X_train_scaled, y_train)

"""
    ده عشان يحفظ المودل في فايل خارجي عشان ميعدش العملية كل مرة وكمان يحفظ طريقة
    السكيل عشان يعمل نفس الطريقة مع الانبوتس الي داخلة
"""
with open('cancer_model.pkl', 'wb') as f:
    pickle.dump(model, f)
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)


Your Top 5 Selection Criteria: ['concave points_worst', 'perimeter_worst', 'concave points_mean', 'radius_worst', 'perimeter_mean']
Files saved: cancer_model.pkl, scaler.pkl


In [ ]:
def predict_probability(inputs):
    model = pickle.load(open('cancer_model.pkl', 'rb'))
    scaler = pickle.load(open('scaler.pkl', 'rb'))
    
    inputs_scaled = scaler.transform([inputs])
    probability = model.predict_proba(inputs_scaled)[0][1] * 100
    return probability

def show_awareness_chart(df, feature_name, user_value):
    plt.figure(figsize=(10, 4))
    sns.kdeplot(data=df, x=feature_name, hue='diagnosis', fill=True, palette='magma')
    plt.axvline(user_value, color='red', linestyle='--', label='Your Value')
    plt.title(f"Awareness: Distribution of {feature_name}")
    plt.legend(['Benign', 'Malignant', 'Your Input'])
    plt.show()

In [ ]:
# for test
print(df.head()) 

show_awareness_chart(df, 'concave points_worst', 0.12)

sample_input = [0.1, 120.0, 0.08, 15.0, 90.0]
prob = predict_probability(sample_input)
print(f"The probability of cancer is: {prob:.2f}%")